# Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo


# Chargement des données


In [ ]:
wine_quality = fetch_ucirepo(id=186)

X = wine_quality.data.features
y = wine_quality.data.targets

df = pd.concat([X, y], axis=1)
df.head()


# Métadonnées et variables


In [ ]:
print(wine_quality.metadata)
print(wine_quality.variables)


# Structure du jeu de données


In [ ]:
print(df.shape)
df.info()


# Statistiques descriptives


In [ ]:
df.describe().round(2)


# Vérification des valeurs manquantes


In [ ]:
df.isna().sum()


# Détection des valeurs aberrantes par l'écart interquartile


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop("quality")
q1 = df[numeric_cols].quantile(0.25)
q3 = df[numeric_cols].quantile(0.75)
iqr = q3 - q1
outliers = ((df[numeric_cols] < (q1 - 1.5 * iqr)) | (df[numeric_cols] > (q3 + 1.5 * iqr))).sum()
outliers


# Proportion des valeurs aberrantes


In [ ]:
(outliers / len(df) * 100).round(2)


# Encodage de la variable catégorielle


In [ ]:
if "color" in wine_quality.data.original.columns:
    df["color"] = wine_quality.data.original["color"].map({"red": 1, "white": 0})
df["color"].value_counts()


# Création de la cible binaire


In [ ]:
df["good_wine"] = (df["quality"] >= 7).astype(int)
df[["quality", "good_wine"]].head()


# Répartition des notes de qualité d'origine


In [ ]:
df["quality"].value_counts().sort_index()


# Distribution de la cible binaire


In [ ]:
counts = df["good_wine"].value_counts()
proportions = (df["good_wine"].value_counts(normalize=True) * 100).round(2)
pd.DataFrame({"effectif": counts, "pourcentage": proportions})


# Visualisation du déséquilibre des classes


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df["quality"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#2b5c8f")
axes[0].set_title("Scores de qualité")
axes[0].set_xlabel("Qualité")
axes[0].set_ylabel("Effectif")

df["good_wine"].value_counts().plot(kind="bar", ax=axes[1], color=["#b83b5e", "#6a2c70"])
axes[1].set_title("Cible binaire")
axes[1].set_xlabel("Bon vin (>= 7)")
axes[1].set_ylabel("Effectif")
axes[1].set_xticklabels(["0 (Ordinaire)", "1 (Bon)"], rotation=0)

plt.tight_layout()
plt.show()


# Matrice de corrélation


In [ ]:
corr_matrix = df.corr().round(2)
corr_matrix


# Corrélations avec la cible binaire


In [ ]:
target_corr = corr_matrix["good_wine"].sort_values(ascending=False)
target_corr


# Visualisation de la matrice de corrélation


In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        plt.text(j, i, corr_matrix.iloc[i, j], ha="center", va="center", color="black", fontsize=7)
plt.title("Matrice de corrélation des variables")
plt.tight_layout()
plt.show()
